# Client Math standalone Avg@12 evaluation

This notebook evaluates an already-trained Math Student on AIME25 and AIME26. Each problem receives 12 independently seeded generations and the reported metric is Avg@12, not pass@12. Generation records are appended to JSONL immediately so interrupted evaluations can resume.

The evaluation generation budget is 38912 new tokens, matching the official OPSD evaluation length. Other decoding settings remain configurable in the Papermill parameters cell.


In [ ]:
# Papermill parameters
GPU_ID = "0"
MODEL_DIR = "results/client_math_opsd_qwen3_4b_fullft/standalone_math_fullft_100steps_single_4090"
OUTPUT_DIR = "results/client_math_opsd_qwen3_4b_fullft/client_math_avg12_evaluation"
ATTN_IMPLEMENTATION = "sdpa"
LOCAL_FILES_ONLY = True
SEED = 42

SAMPLES_PER_PROMPT = 12
MAX_SEQ_LENGTH = 40960
MAX_NEW_TOKENS = 38912
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = -1
MIN_P = 0.0
ENABLE_THINKING = True

LIMIT_AIME25 = None
LIMIT_AIME26 = None
RESUME = True
MAX_CONSECUTIVE_ERRORS = 3

PARALLEL_BENCHMARKS = True
RUN_AIME25 = True
RUN_AIME26 = True
NOTEBOOK_PATH = "client_math_eval.ipynb"


## Runtime and local model


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
if LOCAL_FILES_ONLY:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"

import gc
import hashlib
import json
import random
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import DownloadConfig, disable_progress_bars, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Start the GPU instance and restart the kernel.")
if int(SAMPLES_PER_PROMPT) != 12:
    raise ValueError("This notebook is intended to report Avg@12.")

workspace = Path.cwd().resolve()
model_dir = Path(MODEL_DIR).expanduser()
if not model_dir.is_absolute():
    model_dir = (workspace / model_dir).resolve()
output_dir = Path(OUTPUT_DIR).expanduser()
if not output_dir.is_absolute():
    output_dir = (workspace / output_dir).resolve()
raw_dir = output_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
is_parallel_orchestrator = bool(PARALLEL_BENCHMARKS and RUN_AIME25 and RUN_AIME26)

if not (model_dir / "config.json").is_file():
    raise FileNotFoundError(f"No Transformers model found at {model_dir}")
print(f"Model:  {model_dir}")
print(f"Output: {output_dir}")
print(f"GPU:    {torch.cuda.get_device_name(0)}")
print(f"Mode:   {'parallel AIME25/AIME26 orchestrator' if is_parallel_orchestrator else 'single worker'}", flush=True)


In [ ]:
gc.collect()
torch.cuda.empty_cache()
model = None
tokenizer = None
if is_parallel_orchestrator:
    print("Parent process will not load model weights; each benchmark worker loads one copy.", flush=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, local_files_only=True, trust_remote_code=False
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPLEMENTATION,
        local_files_only=True,
        trust_remote_code=False,
    ).to("cuda:0")
    model.eval()
    model.config.use_cache = True
    model.generation_config.max_length = None
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print(f"CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)


## Locally cached AIME datasets and evaluation fingerprint


In [ ]:
disable_progress_bars()
local_download_config = DownloadConfig(local_files_only=bool(LOCAL_FILES_ONLY))
datasets_to_run = {}
if RUN_AIME25:
    datasets_to_run["aime25"] = load_dataset(
        "math-ai/aime25", split="test", download_config=local_download_config
    )
if RUN_AIME26:
    datasets_to_run["aime26"] = load_dataset(
        "math-ai/aime26", split="test", download_config=local_download_config
    )
if RUN_AIME25 and LIMIT_AIME25 is not None:
    datasets_to_run["aime25"] = datasets_to_run["aime25"].select(
        range(min(int(LIMIT_AIME25), len(datasets_to_run["aime25"])))
    )
if RUN_AIME26 and LIMIT_AIME26 is not None:
    datasets_to_run["aime26"] = datasets_to_run["aime26"].select(
        range(min(int(LIMIT_AIME26), len(datasets_to_run["aime26"])))
    )

weight_files = sorted(model_dir.glob("*.safetensors"))
model_signature_payload = {
    "path": str(model_dir),
    "weights": [
        {"name": path.name, "size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
        for path in weight_files
    ],
}
model_signature = hashlib.sha256(
    json.dumps(model_signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
protocol = {
    "metric": "avg_at_12",
    "samples_per_prompt": SAMPLES_PER_PROMPT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "enable_thinking": ENABLE_THINKING,
    "seed": SEED,
    "model_signature": model_signature,
}
run_fingerprint = hashlib.sha256(
    json.dumps(protocol, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Evaluation fingerprint: {run_fingerprint}")
for name, dataset in datasets_to_run.items():
    print(f"{name}: {len(dataset)} problems, {len(dataset) * SAMPLES_PER_PROMPT} generations")


## Generation and exact-answer utilities


In [ ]:
def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def print_progress(label, done, total, started):
    percentage = 100.0 * done / total if total else 100.0
    print(
        f"[PROGRESS] {label}: {done}/{total} ({percentage:.2f}%) "
        f"elapsed={format_seconds(time.time() - started)}",
        flush=True,
    )


def record_key(sample_id, sample_index):
    return f"{sample_id}::sample-{int(sample_index):02d}"


def stable_seed(benchmark, sample_id, sample_index):
    text = f"{benchmark}:{sample_id}:sample-{sample_index}"
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def load_records(path):
    records = {}
    if not path.is_file():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == run_fingerprint:
                records[record_key(record["sample_id"], record["sample_index"])] = record
    return records


def split_thinking(text):
    text = text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False


def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index]
    return None


def extract_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches:
        return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""


def normalize_answer(value):
    value = str(value).strip().replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value


In [ ]:
def generate_one(problem, benchmark, sample_id, sample_index):
    prompt = (
        str(problem).strip()
        + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
    )
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=bool(ENABLE_THINKING),
    )
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Prompt has {input_tokens} tokens and exceeds MAX_SEQ_LENGTH")
    effective_max_new_tokens = min(int(MAX_NEW_TOKENS), int(available))
    sample_seed = stable_seed(benchmark, sample_id, sample_index)
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=effective_max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=0 if int(TOP_K) < 0 else int(TOP_K),
            min_p=MIN_P,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_thinking(raw_generation)
    return {
        "sample_seed": sample_seed,
        "raw_generation": raw_generation,
        "thinking": thinking,
        "final_answer": final_answer,
        "found_think_close": found_think_close,
        "input_tokens": input_tokens,
        "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": effective_max_new_tokens,
        "generation_seconds": time.time() - started,
    }


## Run or resume 12 generations per problem


In [ ]:
def evaluate_benchmark(benchmark, dataset):
    output_path = raw_dir / f"{benchmark}.jsonl"
    records = load_records(output_path) if RESUME else {}
    completed = {key for key, record in records.items() if record.get("status") == "ok"}
    sample_ids = [str(row.get("id", index)) for index, row in enumerate(dataset)]
    expected = {
        record_key(sample_id, sample_index)
        for sample_id in sample_ids
        for sample_index in range(SAMPLES_PER_PROMPT)
    }
    total = len(expected)
    done = len(expected.intersection(completed))
    if done == total:
        print(f"[SKIP] {benchmark}/avg@12: {total}/{total} generations complete", flush=True)
        return records
    if done:
        print(f"[RESUME] {benchmark}/avg@12: {done}/{total} generations complete", flush=True)

    started = time.time()
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        for sample_index in range(SAMPLES_PER_PROMPT):
            key = record_key(sample_id, sample_index)
            if key in completed:
                continue
            base = {
                "run_fingerprint": run_fingerprint,
                "benchmark": benchmark,
                "sample_id": sample_id,
                "sample_index": sample_index,
                "samples_per_prompt": SAMPLES_PER_PROMPT,
                "evaluation_id": key,
                "problem": str(example["problem"]),
                "gold_answer": str(example["answer"]),
            }
            try:
                generated = generate_one(
                    example["problem"], benchmark, sample_id, sample_index
                )
                predicted = extract_answer(generated["final_answer"])
                record = {
                    **base,
                    **generated,
                    "predicted_answer": predicted,
                    "correct": normalize_answer(predicted) == normalize_answer(example["answer"]),
                    "status": "ok",
                }
                consecutive_errors = 0
            except Exception as exc:
                record = {**base, "correct": False, "status": "error", "error": repr(exc)}
                consecutive_errors += 1
                print(f"[ERROR] {benchmark}:{key}: {exc!r}", flush=True)
                torch.cuda.empty_cache()
            append_jsonl(output_path, record)
            records[key] = record
            done += 1
            print_progress(f"client-math/{benchmark}/avg@12", done, total, started)
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                raise RuntimeError(
                    f"Aborting after {consecutive_errors} consecutive generation errors; "
                    f"last error: {record.get('error')}"
                )

    final_records = load_records(output_path)
    successful = {key for key, record in final_records.items() if record.get("status") == "ok"}
    missing = expected - successful
    if missing:
        raise RuntimeError(f"{benchmark} Avg@12 incomplete: {len(missing)} generations missing or failed")
    return final_records


def launch_parallel_benchmark_workers():
    notebook_path = Path(NOTEBOOK_PATH).expanduser()
    if not notebook_path.is_absolute():
        notebook_path = (workspace / notebook_path).resolve()
    if not notebook_path.is_file():
        raise FileNotFoundError(f"Parallel worker notebook not found: {notebook_path}")

    worker_root = output_dir / "_parallel_workers"
    worker_root.mkdir(parents=True, exist_ok=True)
    common_parameters = {
        "GPU_ID": GPU_ID,
        "MODEL_DIR": str(model_dir),
        "ATTN_IMPLEMENTATION": ATTN_IMPLEMENTATION,
        "LOCAL_FILES_ONLY": LOCAL_FILES_ONLY,
        "SEED": SEED,
        "SAMPLES_PER_PROMPT": SAMPLES_PER_PROMPT,
        "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "TEMPERATURE": TEMPERATURE,
        "TOP_P": TOP_P,
        "TOP_K": TOP_K,
        "MIN_P": MIN_P,
        "ENABLE_THINKING": ENABLE_THINKING,
        "RESUME": RESUME,
        "MAX_CONSECUTIVE_ERRORS": MAX_CONSECUTIVE_ERRORS,
        "PARALLEL_BENCHMARKS": False,
        "NOTEBOOK_PATH": str(notebook_path),
    }
    worker_specs = {
        "aime25": {"RUN_AIME25": True, "RUN_AIME26": False, "LIMIT_AIME25": LIMIT_AIME25},
        "aime26": {"RUN_AIME25": False, "RUN_AIME26": True, "LIMIT_AIME26": LIMIT_AIME26},
    }
    workers = {}
    for benchmark, benchmark_parameters in worker_specs.items():
        worker_dir = worker_root / benchmark
        worker_dir.mkdir(parents=True, exist_ok=True)
        executed_notebook = worker_dir / f"{benchmark}_executed.ipynb"
        log_path = worker_dir / f"{benchmark}.log"
        parameters = dict(common_parameters)
        parameters.update(benchmark_parameters)
        parameters["OUTPUT_DIR"] = str(worker_dir)
        command = [
            sys.executable, "-m", "papermill",
            str(notebook_path), str(executed_notebook),
            "--cwd", str(workspace),
            "--request-save-on-cell-execute",
            "--log-output",
        ]
        command.extend(["--parameters_yaml", json.dumps(parameters)])
        log_handle = log_path.open("w", encoding="utf-8")
        process = subprocess.Popen(
            command, cwd=str(workspace), stdout=log_handle, stderr=subprocess.STDOUT, text=True
        )
        workers[benchmark] = {
            "process": process, "log_handle": log_handle,
            "log_path": log_path, "worker_dir": worker_dir,
        }
        print(f"[LAUNCH] {benchmark}: pid={process.pid}, log={log_path}", flush=True)

    last_report = 0.0
    while any(item["process"].poll() is None for item in workers.values()):
        now = time.time()
        if now - last_report >= 60:
            status = {
                name: ("running" if item["process"].poll() is None else item["process"].returncode)
                for name, item in workers.items()
            }
            print(f"[PARALLEL STATUS] {status}", flush=True)
            last_report = now
        time.sleep(10)

    failures = []
    for benchmark, item in workers.items():
        item["log_handle"].close()
        return_code = item["process"].returncode
        if return_code != 0:
            log_lines = item["log_path"].read_text(encoding="utf-8", errors="replace").splitlines()
            failures.append(
                f"{benchmark} exited with {return_code}:\n" + "\n".join(log_lines[-40:])
            )
        else:
            source_jsonl = item["worker_dir"] / "raw" / f"{benchmark}.jsonl"
            if not source_jsonl.is_file():
                failures.append(f"{benchmark} completed without {source_jsonl}")
            else:
                shutil.copy2(source_jsonl, raw_dir / source_jsonl.name)
                print(f"[COMPLETE] {benchmark}: copied {source_jsonl.name} for aggregation", flush=True)
    if failures:
        raise RuntimeError("\n\n".join(failures))


if is_parallel_orchestrator:
    launch_parallel_benchmark_workers()
    all_records = {
        benchmark: load_records(raw_dir / f"{benchmark}.jsonl")
        for benchmark in datasets_to_run
    }
else:
    all_records = {
        benchmark: evaluate_benchmark(benchmark, dataset)
        for benchmark, dataset in datasets_to_run.items()
    }


## Avg@12 metrics and bar chart


In [ ]:
summary_rows = []
per_problem_rows = []
for benchmark, dataset in datasets_to_run.items():
    records = all_records[benchmark]
    sample_ids = [str(row.get("id", index)) for index, row in enumerate(dataset)]
    for sample_id in sample_ids:
        problem_records = [
            records[record_key(sample_id, sample_index)]
            for sample_index in range(SAMPLES_PER_PROMPT)
        ]
        correct_samples = sum(bool(record["correct"]) for record in problem_records)
        per_problem_rows.append({
            "benchmark": benchmark.upper(),
            "sample_id": sample_id,
            "correct_samples": correct_samples,
            "total_samples": SAMPLES_PER_PROMPT,
            "average_accuracy": correct_samples / SAMPLES_PER_PROMPT,
        })
    benchmark_rows = [row for row in per_problem_rows if row["benchmark"] == benchmark.upper()]
    correct_generations = sum(row["correct_samples"] for row in benchmark_rows)
    total_generations = len(benchmark_rows) * SAMPLES_PER_PROMPT
    summary_rows.append({
        "benchmark": benchmark.upper(),
        "metric": "avg_at_12",
        "problems": len(benchmark_rows),
        "samples_per_prompt": SAMPLES_PER_PROMPT,
        "correct_generations": correct_generations,
        "total_generations": total_generations,
        "score": correct_generations / total_generations,
        "score_percent": 100.0 * correct_generations / total_generations,
    })

summary_df = pd.DataFrame(summary_rows)
per_problem_df = pd.DataFrame(per_problem_rows)
summary_csv = output_dir / "avg_at_12_metrics.csv"
summary_json = output_dir / "avg_at_12_metrics.json"
per_problem_csv = output_dir / "avg_at_12_per_problem.csv"
per_problem_json = output_dir / "avg_at_12_per_problem.json"
chart_path = output_dir / "aime25_aime26_avg_at_12.png"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
per_problem_df.to_csv(per_problem_csv, index=False, encoding="utf-8-sig")
summary_json.write_text(json.dumps({
    "model": str(model_dir),
    "run_fingerprint": run_fingerprint,
    "protocol": protocol,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": summary_rows,
}, ensure_ascii=False, indent=2), encoding="utf-8")
per_problem_json.write_text(
    json.dumps(per_problem_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
labels = summary_df["benchmark"].tolist()
values = summary_df["score_percent"].astype(float).tolist()
figure, axis = plt.subplots(figsize=(7.5, 5.5))
bars = axis.bar(labels, values, color=["#4C78A8", "#72B7B2"], width=0.58,
                edgecolor="#333333", linewidth=0.8)
for bar, value, correct, total in zip(
    bars, values, summary_df["correct_generations"], summary_df["total_generations"]
):
    axis.text(
        bar.get_x() + bar.get_width() / 2, value + 1.2,
        f"{value:.2f}%\n({correct}/{total} generations)",
        ha="center", va="bottom", fontsize=10,
    )
axis.set_ylabel("Avg@12 exact-match accuracy (%)")
axis.set_ylim(0, 108)
axis.set_title("Math Student - AIME Avg@12")
axis.grid(axis="y", linestyle="--", alpha=0.3)
figure.tight_layout()
figure.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

display(summary_df)
print(f"Summary JSON:    {summary_json}")
print(f"Summary CSV:     {summary_csv}")
print(f"Per-problem CSV: {per_problem_csv}")
print(f"Bar chart:       {chart_path}", flush=True)

del model
gc.collect()
torch.cuda.empty_cache()


## Interpretation

Avg@12 is the mean correctness over 12 generations per problem. A problem solved once in 12 attempts contributes `1/12`; it is not counted as fully solved as it would be under pass@12. For a 30-problem AIME benchmark this notebook evaluates 360 generations.

This notebook matches the project's decoding settings, not the paper's complete decoding protocol. To reproduce the paper protocol, thinking, temperature, and the generation budget must also be changed.
